Training process :- 

SImilar to other Neural networks

Add training data

Calculate Loss

Backprop through model

update weights

Hyper parameters

1. Learning rate

2. Learning rate Scheduler

3. Optimizer hyperparameters

General example of pytorch for training

for epoch in range(num_epochs):

      for batch in train_dataloader:

          outputs = model(**batch)

          loss = outputs.loss

          loss.backward()

          optimizer.step()

In [15]:
import datasets
import tempfile
import logging
import random
import config
import os
import yaml
import logging
import time
import torch
import transformers

# from utilities import *
from utilities import tokenize_and_split_data
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM
from transformers import TrainingArguments
from lamini import Lamini

ModuleNotFoundError: No module named 'utilities'

In [5]:
logger = logging.getLogger(__name__)
global_config = None

In [6]:
# Loading dataset without using hugging face, using locally stored dataset

dataset_name = "lamini_docs.jsonl"
dataset_path = "/Users/apple/Desktop/project/LLM/Finetuning_llm/Data/lamini_docs.jsonl"
use_hf = False

In [7]:
dataset_name

'lamini_docs.jsonl'

In [8]:
# Loading dataset using hugging face

# dataset_path = "lamini/lamini_docs"
# use_hf=True

In [9]:
model_name = "EleutherAI/pythia-70m"

Setup the model, training config, and tokenizer

In [75]:
training_config = {
    "model": {
        "pretrained_name": model_name,
        "max_length":2048
    },
    "dataset": {
        "use_hf":use_hf,
        "path": dataset_path
    },
    "verbose": True
}

In [88]:
# import pandas as pd
# instruction_dataset_df = pd.read_json("/Users/apple/Desktop/project/LLM/Finetuning_llm/Data/lamini_docs.jsonl", lines=True)
# instruction_dataset_df

In [89]:
# examples = instruction_dataset_df.to_dict()
# text = examples["question"][0]+examples['answer'][0]
# text

In [90]:
# if "question" in examples and "answer" in examples:
#     text = examples["question"][0] + examples['answer'][0]
# elif "instruction" in examples and "response" in examples:
#     text = examples["instructions"][0] + examples['response'][0]
# elif "input" in examples and "output" in examples:
#     text = examples["input"][0] + examples['output'][0] 
# else:
#     text = examples["text"][0]
    



In [91]:
# prompt_template_qa = """### Question:
# {question}

# ### Answer:
# {answer}"""

In [92]:
# question = examples["question"][0]
# answer = examples["answer"][0]

# text_with_prompt_template = prompt_template_qa.format(question=question, answer=answer)
# text_with_prompt_template

In [93]:
# prompt_template_q = """### Question:
# {question}

# ### Answer:"""

In [94]:
num_examples = len(examples["question"])
finetuning_dataset_text_only=[]
finetuning_dataset_question_answer = []
for i in range(num_examples):
    question = examples["question"][i]
    answer = examples["answer"][i]
    
    text_with_prompt_template_qa = prompt_template_qa.format(question=question,answer=answer)
    finetuning_dataset_text_only.append({"text":text_with_prompt_template_qa})
    
    text_with_prompt_template_q = prompt_template_q.format(question=question)
    finetuning_dataset_question_answer.append({"question":text_with_prompt_template_q, "answer": answer})

In [95]:
#  # Split data
# def tokenize_and_split_data(tokenizer):
#     test_size=0.1
#     train_data, test_data = train_test_split(text_with_prompt_template_q, test_size=test_size, random_state=42)

#     def tokenize_function(examples):
#         inputs = tokenizer(
#             examples["question"],  # ✅ this is now a list of strings
#             padding="max_length",
#             truncation=True,
#             max_length=config["model"]["max_length"],
#         )
#         outputs = tokenizer(
#             examples["answer"],
#             padding="max_length",
#             truncation=True,
#             max_length=config["model"]["max_length"],
#         )
#         inputs["labels"] = outputs["input_ids"]
#         return inputs


#     train_dataset = Dataset.from_list(train_data).map(tokenize_function, batched=True)
#     test_dataset = Dataset.from_list(test_data).map(tokenize_function, batched=True)

#     return train_dataset, test_dataset


In [96]:
import json
from datasets import Dataset
from sklearn.model_selection import train_test_split

prompt_template = """### Question:
{question}

### Answer:"""

def tokenize_and_split_data(config, tokenizer):
    dataset_info = config["dataset"]

    # Load raw data from JSONL
    path = dataset_info.get("path")
    if path is None:
        raise ValueError("Missing dataset path in config['dataset']['path']")

    with open(path, "r") as f:
        raw_data = [json.loads(line) for line in f]

    # Build structured question-answer pairs with prompt formatting
    qa_data = []
    for item in raw_data:
        question = item.get("question", "")
        answer = item.get("answer", "")
        qa_data.append({
            "question": prompt_template.format(question=question),
            "answer": answer
        })

    # Split into train/test
    test_size = config.get("test_size", 0.1)
    train_data, test_data = train_test_split(qa_data, test_size=test_size, random_state=42)

    # Tokenize function (expects dict of lists from HF Datasets)
    def tokenize_function(examples):
        inputs = tokenizer(
            examples["question"],
            padding="max_length",
            truncation=True,
            max_length=config["model"]["max_length"],
        )
        outputs = tokenizer(
            examples["answer"],
            padding="max_length",
            truncation=True,
            max_length=config["model"]["max_length"],
        )
        inputs["labels"] = outputs["input_ids"]
        return inputs

    # Create Dataset objects
    train_dataset = Dataset.from_list(train_data).map(tokenize_function, batched=True)
    test_dataset = Dataset.from_list(test_data).map(tokenize_function, batched=True)

    return train_dataset, test_dataset

In [97]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
train_dataset, test_dataset = tokenize_and_split_data(training_config, tokenizer)

print(train_dataset)
print(test_dataset)

Map: 100%|██████████| 140/140 [00:00<00:00, 1908.29 examples/s]

Dataset({
    features: ['question', 'answer', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 1260
})
Dataset({
    features: ['question', 'answer', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 140
})


In [98]:
base_model = AutoModelForCausalLM.from_pretrained(model_name)

In [101]:
device_count = torch.cuda.device_count()
if device_count > 0:
    logger.debug("Select GPU device")
    device = torch.device("cuda")
else:
    logger.debug("Select CPU device")
    device = torch.device("cpu")

In [103]:
base_model.to(device)

GPTNeoXForCausalLM(
  (gpt_neox): GPTNeoXModel(
    (embed_in): Embedding(50304, 512)
    (emb_dropout): Dropout(p=0.0, inplace=False)
    (layers): ModuleList(
      (0-5): 6 x GPTNeoXLayer(
        (input_layernorm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (post_attention_layernorm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (post_attention_dropout): Dropout(p=0.0, inplace=False)
        (post_mlp_dropout): Dropout(p=0.0, inplace=False)
        (attention): GPTNeoXAttention(
          (query_key_value): Linear(in_features=512, out_features=1536, bias=True)
          (dense): Linear(in_features=512, out_features=512, bias=True)
        )
        (mlp): GPTNeoXMLP(
          (dense_h_to_4h): Linear(in_features=512, out_features=2048, bias=True)
          (dense_4h_to_h): Linear(in_features=2048, out_features=512, bias=True)
          (act): GELUActivation()
        )
      )
    )
    (final_layer_norm): LayerNorm((512,), eps=1e-05, elementwise